# LangChain: Chains

## LLMChain
LLMChain es una cadena básica que involucra a un modelo de lenguaje grande (Large Language Model, LLM). Esta cadena toma una entrada, la procesa a través del LLM, y produce una salida. Es la estructura fundamental en LangChain para construir aplicaciones que interactúan con modelos de lenguaje.
### Características:
* Utiliza un LLM para generar respuestas.
* Puede ser utilizado como un bloque de construcción para cadenas más complejas.

## Sequential Chains
Las cadenas secuenciales permiten encadenar múltiples LLMChain o otras cadenas de manera secuencial. Hay dos tipos principales:

## SimpleSequentialChain
SimpleSequentialChain es una cadena donde la salida de una cadena se utiliza como entrada para la siguiente. Es adecuada para flujos de trabajo simples donde cada paso depende del anterior.
### Características:
* Encadena múltiples pasos de forma secuencial.
* Cada paso toma la salida del anterior como su entrada.
* Fácil de configurar para flujos de trabajo lineales.

### Ejemplo:
#### Cadena 1: Genera una lista de temas a partir de una pregunta.
#### Cadena 2: Toma la lista de temas y genera una breve descripción para cada uno.

## SequentialChain
SequentialChain es más flexible que SimpleSequentialChain y permite manejar múltiples entradas y salidas en cada paso. Puede realizar operaciones más complejas que dependen de múltiples entradas o producen múltiples salidas.
### Características:

* Permite pasos con múltiples entradas y salidas.
* Adecuada para flujos de trabajo más complejos.
* Más flexible y configurable que SimpleSequentialChain.
### Ejemplo:
#### Cadena 1: Genera una lista de temas a partir de una pregunta.
#### Cadena 2: Toma la lista de temas y genera una breve descripción para cada uno.
#### Cadena 3: Toma las descripciones y genera un resumen general.

## Router Chain
Router Chain es una cadena que puede dirigir la entrada a diferentes sub-cadenas basándose en alguna lógica de enrutamiento. Es útil cuando tienes múltiples posibles flujos de trabajo y necesitas decidir cuál ejecutar en tiempo de ejecución.

### Características:
* Dirige la entrada a diferentes cadenas basándose en reglas definidas.
* Adecuado para flujos de trabajo donde se necesita lógica de decisión.
* Permite manejar entradas dinámicas y rutas de procesamiento variadas.
### Ejemplo:
#### Recibe una solicitud.
#### Analiza la solicitud para determinar su tipo (e.g., pregunta técnica, consulta de ventas).
#### Redirige la solicitud a la cadena adecuada (e.g., cadena de soporte técnico, cadena de ventas).

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
llm_model = "gpt-3.5-turbo"

In [3]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [4]:
df.head()

,Product,Review
0,Sofá Clásico,"Muy cómodo y elegante, se ve genial en mi sala."
1,Lámpara de Pie,Ilumina muy bien y tiene un diseño moderno que...
2,Mesa de Centro,"Perfecta para mi sala, el tamaño y el diseño s..."
3,Silla de Comedor,"Muy cómoda y robusta, ideal para largas cenas."
4,Estantería Moderna,Gran capacidad de almacenamiento y fácil de en...


## LLMChain

In [5]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [6]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [7]:
prompt = ChatPromptTemplate.from_template(
    "¿Cuál es el mejor nombre para describir \
    una empresa que fabrica {producto}?"
)

In [8]:
chain = llm | prompt

In [9]:
producto = "Juego de Sábanas Queen Size"
chain.invoke(producto)

ChatPromptValue(messages=[HumanMessage(content="¿Cuál es el mejor nombre para describir     una empresa que fabrica content='Las sábanas Queen Size son ideales para camas de tamaño estándar que miden aproximadamente 152 cm x 203 cm. Un juego de sábanas Queen Size generalmente incluye una sábana ajustable, una sábana plana y dos fundas de almohada.\\n\\nAl elegir un juego de sábanas Queen Size, es importante considerar el material y la calidad de las sábanas. Algunas opciones populares incluyen el algodón, el poliéster, la microfibra y el satén. También puedes optar por sábanas con diferentes tipos de hilado, como el percal, la sábana, el satén o el jersey.\\n\\nAdemás, asegúrate de verificar las dimensiones del juego de sábanas para asegurarte de que se ajusten adecuadamente a tu colchón Queen Size. También puedes elegir entre una variedad de colores y estampados para que coincidan con la decoración de tu dormitorio.\\n\\nEn resumen, un juego de sábanas Queen Size es una excelente opci

## SimpleSequentialChain

In [10]:
from langchain.chains import LLMChain, SimpleSequentialChain

In [35]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

first_prompt = ChatPromptTemplate.from_template(
    "¿Cuál es el mejor nombre para describir \
    una empresa que fabrica {producto}?"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [12]:
second_prompt = ChatPromptTemplate.from_template(
    "Escribe una descripción de 20 palabras para la siguiente \
    empresa: {nombre_de_empresa}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [13]:
overall_simple_chain = SimpleSequentialChain(
    chains=[chain_one, chain_two],
    verbose=True
)

In [14]:
overall_simple_chain.invoke(producto)



> Entering new SimpleSequentialChain chain...
"Royal Dreams Linens"
Empresa especializada en la fabricación y venta de lujosos y confortables juegos de sábanas, cobertores y almohadas de alta calidad.

> Finished chain.


{'input': 'Juego de Sábanas Queen Size',
 'output': 'Empresa especializada en la fabricación y venta de lujosos y confortables juegos de sábanas, cobertores y almohadas de alta calidad.'}

## SequentialChain

In [15]:
from langchain.chains import SequentialChain

In [16]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(
    llm=llm, 
    prompt=first_prompt, 
    output_key="English_Review"
)

In [17]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(
    llm=llm, 
    prompt=second_prompt, 
    output_key="summary"
)

In [18]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(
    llm=llm, 
    prompt=third_prompt,
    output_key="language"
)

In [19]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(
    llm=llm, 
    prompt=fourth_prompt,
    output_key="followup_message"
)

In [20]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

In [21]:
review = df.Review[5]
overall_chain.invoke(review)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': 'Añade un toque elegante a la habitación, la luz es perfecta.',
 'English_Review': 'It adds an elegant touch to the room, the light is perfect.',
 'summary': 'The reviewer found the product to add an elegant touch to the room and provide perfect lighting.',
 'followup_message': '¡Gracias por tus amables palabras! Nos alegra saber que nuestro producto ha añadido un toque elegante a tu habitación y que la iluminación es perfecta. ¡Esperamos que sigas disfrutando de nuestros productos en el futuro!'}

## Router Chain

In [22]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [23]:
llm = ChatOpenAI(temperature=0, model=llm_model)

In [24]:
physics_template = """
Eres un profesor de física muy inteligente.
Eres excelente respondiendo preguntas sobre física de manera concisa y fácil de entender.
Cuando no sabes la respuesta a una pregunta, admites que no la sabes.

Aquí hay una pregunta:
{input}"""


math_template = """Eres un muy buen matemático.
Eres excelente respondiendo preguntas de matemáticas.
Eres tan bueno porque eres capaz de descomponer
problemas difíciles en sus partes componentes,
responder las partes componentes y luego juntarlas
para responder la pregunta más amplia.

Aquí hay una pregunta:
{input}"""

history_template = """Eres un muy buen historiador.
Tienes un excelente conocimiento y comprensión de las personas,
eventos y contextos de una variedad de períodos históricos.
Tienes la capacidad de pensar, reflexionar, debatir, discutir y
evaluar el pasado. Tienes respeto por la evidencia histórica
y la capacidad de utilizarla para apoyar tus explicaciones
y juicios.

Aquí hay una pregunta:
{input}"""


computerscience_template = """Eres un exitoso científico de la computación.
Tienes una pasión por la creatividad, la colaboración,
el pensamiento innovador, la confianza, fuertes capacidades para resolver problemas,
comprensión de teorías y algoritmos, y excelentes habilidades de comunicación.
Eres excelente respondiendo preguntas de programación.
Eres tan bueno porque sabes cómo resolver un problema describiendo la solución en pasos imperativos
que una máquina puede interpretar fácilmente y sabes cómo
elegir una solución que tenga un buen equilibrio entre
complejidad temporal y complejidad espacial.

Aquí hay una pregunta:
{input}"""

In [25]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [26]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [27]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [28]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Dado un texto de entrada sin procesar para un
modelo de lenguaje, selecciona el prompt del modelo que mejor se ajuste a la entrada.
Se te proporcionarán los nombres de los prompts disponibles y una
descripción de para qué está mejor adaptado cada prompt.
También puedes revisar la entrada original si crees que revisarla
llevará a una mejor respuesta del modelo de lenguaje.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string  name of the prompt to use or "DEFAULT"
    "next_inputs": string  a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt
names specified below OR it can be "DEFAULT" if the input is not
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [29]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [30]:
chain = MultiPromptChain(
    router_chain=router_chain, 
    destination_chains=destination_chains, 
    default_chain=default_chain, verbose=True
)

In [31]:
chain.invoke("¿Qué es la radiación del cuerpo negro?")



> Entering new MultiPromptChain chain...
physics: {'input': '¿Qué es la radiación del cuerpo negro?'}
> Finished chain.


{'input': '¿Qué es la radiación del cuerpo negro?',
 'text': 'La radiación del cuerpo negro es la radiación electromagnética emitida por un objeto que absorbe toda la radiación que incide sobre él. Se caracteriza por tener un espectro continuo de frecuencias que depende únicamente de la temperatura del objeto. Este fenómeno es importante en la física cuántica y en la teoría de la relatividad.'}

In [32]:
chain.invoke("Cuanto es 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'Cuanto es 2 + 2'}
> Finished chain.


{'input': 'Cuanto es 2 + 2', 'text': 'La respuesta es 4.'}

In [33]:
chain.invoke("Quien es Manuel Belgrano?")



> Entering new MultiPromptChain chain...
History: {'input': 'Quien es Manuel Belgrano?'}
> Finished chain.


{'input': 'Quien es Manuel Belgrano?',
 'text': 'Manuel Belgrano fue un abogado, economista, periodista, político y militar argentino que desempeñó un papel importante en la lucha por la independencia de Argentina. Fue uno de los líderes de la Revolución de Mayo de 1810 y creó la bandera argentina, conocida como la Bandera de Belgrano. Belgrano también fue un defensor de la educación y la igualdad de derechos, y es considerado uno de los padres de la patria argentina.'}

In [34]:
chain.invoke("Cual es el rol del Scheduler y del Dispatcher en un SO")



> Entering new MultiPromptChain chain...
computer science: {'input': 'Cual es el rol del Scheduler y del Dispatcher en un SO'}
> Finished chain.


{'input': 'Cual es el rol del Scheduler y del Dispatcher en un SO',
 'text': 'El Scheduler y el Dispatcher son dos componentes clave en un sistema operativo (SO) que trabajan juntos para administrar la ejecución de procesos.\n\nEl Scheduler es responsable de decidir qué proceso se ejecutará a continuación en la CPU. Utiliza diferentes políticas de planificación para tomar esta decisión, como la planificación por prioridad, la planificación por tiempo compartido, entre otras. El Scheduler se encarga de asignar recursos de manera eficiente y equitativa entre los procesos en ejecución.\n\nPor otro lado, el Dispatcher es el encargado de llevar a cabo el cambio de contexto entre procesos. Cuando el Scheduler decide que un proceso debe ejecutarse, el Dispatcher se encarga de detener el proceso actual, guardar su estado en la memoria, cargar el estado del nuevo proceso en la CPU y comenzar su ejecución. El Dispatcher es crucial para garantizar una transición suave y eficiente entre procesos.\